# 1. Setup and installation

In [ ]:
%pip install -e libs/BERTopic-master
%pip install setfit==1.0.3
%pip install transformers==4.41.2 tokenizers==0.19.1
%pip install -e libs/setfit-main
%pip install sentence-transformers umap-learn scikit-learn pandas pyarrow nltk gensim

In [ ]:
import nltk

try:
    import torch
    cuda_available = torch.cuda.is_available()
except ImportError:
    cuda_available = False
    print("torch is not installed; CUDA availability cannot be checked via torch.")

print(f"CUDA available: {cuda_available}")
nltk.download("punkt")
nltk.download("punkt_tab", quiet=True)

# 2. Segmentation

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import segment

segments = segment.run_segmentation(
    raw_dir=project_root / "data" / "raw",
    output_path=project_root / "data" / "interim" / "segments.parquet",
)
segments.head()

# 3. BERTopic – unsupervised theme discovery

In [ ]:
import importlib, bertopic_run
importlib.reload(bertopic_run)
topic_membership = bertopic_run.run_bertopic(
    segments_path=project_root / "data" / "interim" / "segments.parquet",
    output_dir=str(project_root / "data" / "processed"),
)
topic_membership.head()

In [ ]:
bertopic_run.update_topic_representation(
    segments_path=project_root / "data" / "interim" / "segments.parquet",
    model_path=str(project_root / "data" / "processed" / "bertopic_model"),
)

In [ ]:
import importlib, export_topic_words
importlib.reload(export_topic_words)
export_topic_words.run(
    model_path=str(project_root / "data" / "processed" / "bertopic_model"),
    segments_path=project_root / "data" / "interim" / "segments.parquet",
    output_path=str(project_root / "data" / "processed" / "topic_words.xlsx"),
)

In [ ]:
# 3.5 — BERTopic Validation
import importlib, bertopic_validation
importlib.reload(bertopic_validation)
bertopic_validation.run_validation(
    model_path   = str(project_root / "data" / "processed" / "bertopic_model"),
    segments_path= project_root / "data" / "interim" / "segments.parquet",
    topic_path   = project_root / "data" / "processed" / "topic_membership.csv",
    labels_path  = project_root / "labels" / "condition_labels.csv",
    output_dir   = str(project_root / "data" / "processed"),
)

# 4. SetFit – supervised condition coding (founder side)

In [ ]:
import importlib, setfit_conditions
importlib.reload(setfit_conditions)
setfit_conditions.smoke_test(
    labels_path=project_root / "labels" / "condition_labels.csv",
    segments_path=project_root / "data" / "interim" / "segments.parquet",
    output_dir=str(project_root / "data" / "processed"),
)

In [ ]:
import importlib, setfit_conditions
importlib.reload(setfit_conditions)
conditions_proba = setfit_conditions.run_setfit_conditions(
    labels_path=project_root / "labels" / "condition_labels.csv",
    segments_path=project_root / "data" / "interim" / "segments.parquet",
    output_dir=str(project_root / "data" / "processed"),
)
conditions_proba.head()

# 5. SetFit – supervised feedback type coding (evaluator side)

# 6. Aggregation and calibration → qca_calibrated.csv

In [ ]:
import importlib, calibrate
importlib.reload(calibrate)
qca_input = calibrate.run_calibration(
    batch3_path  = project_root / "labels" / "Batch_3_CF_Coding.xlsx",
    batch4_path  = project_root / "labels" / "Batch_4_CF_Coding.xlsx",
    batch5_path  = project_root / "labels" / "Batch_5_CF_Coding.xlsx",
    condition_labels_path = project_root / "labels" / "condition_labels.csv",
    feedback_labels_path  = project_root / "labels" / "feedback_labels.csv",
    topic_path   = project_root / "data" / "processed" / "topic_membership.csv",
    output_path  = project_root / "data" / "processed" / "qca_calibrated.csv",
    r_script_path = project_root / "qca" / "fsqca.R",
)
print(qca_input.shape)
qca_input.head()